In [0]:
%run ../../../utils/pipeline_utils

In [0]:
# Databricks notebook source
# =============================================================================
# common/prime_dim_ia.py
# Primes assist_dev.common.dim_ia
#
# Source tables (Silver):
#   silver_aasbs_ia             → base IA record (piid is the IA number)
#   silver_aasbs_lu_ia_status   → ia_status_desc decoded
#   silver_aasbs_lu_instrument_type (if exists) or lu_ia_type → instrument/type desc
#
# Grain       : One row per ia.id (each IA = one current row on prime)
# SCD Type    : 2 — eff_start_dt=NOW, eff_end_dt=NULL, is_current_flag=TRUE
# Idempotent  : YES — TRUNCATE then INSERT
# Dependencies: dim_agency (for servicing_agency_sk and requesting_agency_sk)
#
# Source column notes:
#   ia.id                    → ia_id (natural key)
#   ia.piid                  → ia_num (the IA PIID / agreement number)
#   ia.ia_status_cd          → ia_status_cd
#   ia.instrument_type_cd    → instrument_type_cd (MIPR=M, Economy Act=S, etc.)
#   ia.fiscal_year           → fiscal_year
#   ia.ia_began_dt           → ia_start_dt
#   ia.activity_address_cd   → servicing AAC → join to dim_agency
#   ia.serv_agency_cd        → servicing agency internal code (backup AAC join)
#   ia.latest_ia_amendment_id → used in delta refresh for amendment tracking
#   NOTE: ia_type_cd is not a separate field on ia; instrument_type_cd is used
#         as the closest equivalent.  A lu_instrument_type lookup decodes it.
#         requesting_agency_sk is left NULL at prime; populated via LOA/funding
#         in delta refresh once client agency association is established.
# =============================================================================

# COMMAND ----------
# MAGIC %run ../utils/pipeline_utils
# COMMAND ----------
dbutils.widgets.text("run_id",   "", "Pipeline Run ID")
dbutils.widgets.text("job_name", "dp1_prime_full", "Job Name")

RUN_ID   = dbutils.widgets.get("run_id")   or "manual-" + get_spark_app_id()
JOB_NAME = dbutils.widgets.get("job_name")

TARGET     = gold("common", "dim_ia")
TASK       = "prime_dim_ia"

S_IA       = silver("aasbs", "ia")
S_STATUS   = silver("aasbs", "lu_ia_status")
# lu_instrument_type is not a confirmed separate table in source DDL;
# instrument_type_cd values are decoded inline with CASE expression.
G_AGENCY   = gold("common", "dim_agency")

print(f"[{TASK}] target={TARGET}")

# COMMAND ----------
start_ts = audit_start(spark, RUN_ID, JOB_NAME, TASK, TARGET,
                        source_schema="aasbs", source_table="ia")

# COMMAND ----------
try:
    truncate_gold(spark, TARGET)

    # ia_sk is GENERATED ALWAYS AS IDENTITY — excluded from INSERT col list.
    spark.sql(f"""
        INSERT INTO {TARGET}
        (
            ia_id, ia_num,
            ia_type_cd, ia_type_desc,
            ia_status_cd, ia_status_desc,
            instrument_type_cd, instrument_type_desc,
            fiscal_year,
            ia_start_dt, ia_end_dt,
            total_direct_cost_est_amt, total_charges_est_amt,
            servicing_agency_sk, requesting_agency_sk,
            program_cd, region_cd,
            eff_start_dt, eff_end_dt, is_current_flag,
            _gold_created_at, _gold_updated_at, _source_batch_id
        )
        WITH ia_base AS (
            SELECT
                ia.id                               AS ia_id,
                -- piid is the IA agreement number / PIID in ASSIST
                ia.piid                             AS ia_num,
                -- ia_type is not a distinct field; instrument_type stands in
                ia.instrument_type_cd               AS ia_type_cd,
                ia.instrument_type_cd               AS instrument_type_cd,
                ia.ia_status_cd,
                ia.fiscal_year,
                ia.ia_began_dt                      AS ia_start_dt,
                -- ia end date: not stored directly on ia; NULL here,
                -- populated from IA amendment ginv_pop_end_dt in delta refresh
                CAST(NULL AS TIMESTAMP)             AS ia_end_dt,
                -- Cost estimates: not directly on ia row; sourced from
                -- funding_amendment totals in delta refresh. NULL on prime.
                CAST(NULL AS DECIMAL(15,2))         AS total_direct_cost_est_amt,
                CAST(NULL AS DECIMAL(15,2))         AS total_charges_est_amt,
                -- AAC used to resolve servicing_agency_sk
                ia.activity_address_cd              AS serv_aac,
                ia.serv_agency_cd,
                -- Program and region come from activity_address_code table
                -- activity_address_cd.program_cd and region_cd
                -- joined via the AAC
                ia.activity_address_cd
            FROM {S_IA} ia
            WHERE NVL(ia.is_deleted,FALSE) = FALSE
        ),
        with_status AS (
            SELECT
                ib.*,
                COALESCE(st.description, ib.ia_status_cd)  AS ia_status_desc
            FROM ia_base ib
            LEFT JOIN {S_STATUS} st
                ON ib.ia_status_cd = st.cd
                AND NVL(st.is_deleted,FALSE) = FALSE
        ),
        with_sk AS (
            SELECT
                ws.*,
                -- Resolve servicing_agency_sk via AAC match
                ag.agency_sk                        AS servicing_agency_sk,
                ag.agency_code,
                -- Requesting agency: not directly on IA — NULL on prime
                CAST(NULL AS BIGINT)                AS requesting_agency_sk,
                -- Program and region from dim_agency AAC attributes
                ag.aac_description
            FROM with_status ws
            LEFT JOIN {G_AGENCY} ag
                ON ws.serv_aac = ag.activity_address_cd
                AND NVL(ag.is_current_flag,FALSE) = TRUE
        )
        SELECT
            ia_id,
            ia_num,
            -- ia_type_cd: map instrument type to broader IA type bucket
            CASE instrument_type_cd
                WHEN 'M'  THEN 'MIPR'
                WHEN 'S'  THEN 'ECONOMY_ACT'
                WHEN 'GT' THEN 'GTC'
                ELSE instrument_type_cd
            END                                     AS ia_type_cd,
            -- ia_type_desc: decoded instrument type description
            CASE instrument_type_cd
                WHEN 'M'  THEN 'Military Interdepartmental Purchase Request'
                WHEN 'S'  THEN 'Economy Act Order (31 U.S.C. 1535)'
                WHEN 'GT' THEN 'Governmentwide Acquisition Contract'
                ELSE COALESCE(instrument_type_cd, 'Unknown')
            END                                     AS ia_type_desc,
            ia_status_cd,
            ia_status_desc,
            instrument_type_cd,
            -- instrument_type_desc
            CASE instrument_type_cd
                WHEN 'M'  THEN 'MIPR'
                WHEN 'S'  THEN 'Economy Act'
                WHEN 'GT' THEN 'GT&C'
                WHEN 'P'  THEN 'Participating Agency'
                ELSE COALESCE(instrument_type_cd, 'Unknown')
            END                                     AS instrument_type_desc,
            fiscal_year,
            ia_start_dt,
            ia_end_dt,
            total_direct_cost_est_amt,
            total_charges_est_amt,
            servicing_agency_sk,
            requesting_agency_sk,
            -- Program and region: sourced from lu_activity_address_code via AAC.
            -- The AAC is on the ia record as activity_address_cd.
            -- We read program_cd and region_cd from Silver AAC table in the view below.
            aac.program_cd                          AS program_cd,
            aac.region_cd                           AS region_cd,
            -- SCD2 initial values
            CURRENT_TIMESTAMP()                     AS eff_start_dt,
            CAST(NULL AS TIMESTAMP)                 AS eff_end_dt,
            TRUE                                    AS is_current_flag,
            CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP(), '{RUN_ID}'
        FROM with_sk ws
        -- Pull program_cd and region_cd from the AAC Silver table
        LEFT JOIN {silver("aasbs", "lu_activity_address_code")} aac
            ON ws.serv_aac = aac.cd
            AND NVL(aac.is_deleted,FALSE) = FALSE
    """)

    n = row_count(spark, TARGET)
    print(f"  [OK] Inserted {n:,} IA rows")

    # Verify no IA has a NULL ia_num (piid is NOT NULL in source)
    null_piid = spark.sql(
        f"SELECT COUNT(*) AS n FROM {TARGET} WHERE ia_num IS NULL"
    ).collect()[0]["n"]
    if null_piid > 0:
        print(f"  [WARN] {null_piid} IAs have NULL ia_num — check silver_aasbs_ia.piid")

    audit_success(spark, RUN_ID, TARGET, n, n, start_ts)

except Exception as e:
    audit_fail(spark, RUN_ID, TARGET, str(e), traceback.format_exc(), start_ts)
    raise

# COMMAND ----------
dbutils.notebook.exit("SUCCESS")
